# How fragile is statistical evidence?

## Missing data, response contamination and sample size

The first notebook demonstrated that covariance geometry, inferential significance, prediction and decision value are distinct.

This second notebook asks what survives when data collection is imperfect.

We compare six fixed regression pipelines and two training-only selection rules under:

1. **Clean predictors and outcomes**, our paired baseline.
2. **MCAR predictors**, missing completely at random.
3. **MAR predictors**, where the probability of missingness depends on another *observed* predictor.
4. **Training response outliers**, gross recording errors in 10% of training outcomes.
5. **MAR plus response outliers**, a combined perturbation.

We vary training sample size, repeat independent simulations, and evaluate models on an independent test set. Across perturbations within the same replication, the underlying original dataset stays unchanged.

### What is measured?

Let \(m(x)=E[Y\mid X=x]\), \(c\) be the fixed intervention cost, and \(\hat m\) the model. We take action when \(\hat m(x)>c\). The test evaluation compares each action with an oracle that knows the *full-data* conditional mean:

\[
R(\hat m)=E[(m(X)-c)_+]-E[(m(X)-c)\mathbf{1}\{\hat m(X_{\mathrm{obs}})>c\}].
\]

Because the oracle sees information missing from \(X_{\mathrm{obs}}\), regret in MCAR/MAR cases **includes irreducible information loss**. Differences in regret between two models in the same stress condition are more directly comparable than an absolute regret across conditions.

We also report observed test RMSE. Test conditional means and test outcomes do not enter fitting or validation selection.

This is a prediction-and-utility study. It is *not* evidence that intervening on a covariate would cause a benefit.

In [ ]:
from __future__ import annotations

import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from analysis import simulate_data
from stress_analysis import (
    STRESSES,
    apply_stress,
    paired_regret_shift,
    run_stress_experiment,
)

REPEATS: int = int(os.environ.get("STRESS_REPEATS", "6"))
TRAIN_SIZES: tuple[int, ...] = tuple(
    int(size.strip())
    for size in os.environ.get("STRESS_SIZES", "80,180,360").split(",")
)
COST: float = 0.6

if REPEATS < 1 or not TRAIN_SIZES or any(n < 64 for n in TRAIN_SIZES):
    raise ValueError("STRESS_REPEATS must be positive and all STRESS_SIZES >= 64")
print(f"Replicates: {REPEATS}; sample sizes: {TRAIN_SIZES}; cost: {COST}")

## 1. Controlled data failures

The same clean data-generating process as in the first notebook supplies 16 correlated predictors and two possible response mechanisms. One depends on a **low-variance contrast**, the other on a **high-variance sum**.

We deliberately leave \(X_3\) fully observed, so it can serve as an anchor for an identifiable MAR missingness process:

\[
P(M_j=1\mid X)=0.20+0.15\tanh(X_3), \quad j\ne 3.
\]

For MCAR, \(P(M_j=1)=0.20\) for the same columns. The anchor is always observed in both cases. Training outcome outliers are generated by adding \(\pm12\) units to 10% of labels, without changing predictors or the true test outcomes.

This distinguishes predictable missingness from contamination of the training target, rather than conflating all failures.

In [ ]:
baseline = simulate_data(seed=2026, n_train=600, n_test=120)
stress_preview = [
    apply_stress(baseline, stress=stress, seed=44)
    for stress in STRESSES
]
observed_rates = pd.DataFrame(
    [
        {
            "stress": stress,
            "training_X_missing": data.missing_train_rate,
            "test_X_missing": data.missing_test_rate,
            "training_Y_corrupted": data.contaminated_train_rate,
        }
        for stress, data in zip(STRESSES, stress_preview)
    ]
)
display(observed_rates.round(3))

In [ ]:
anchor = baseline.x_train[:, 2]
fig, ax = plt.subplots(figsize=(8, 4))
for name in ("mcar", "mar"):
    observed = apply_stress(baseline, name, seed=44)
    data = pd.DataFrame(
        {
            "anchor": anchor,
            "missing_X1": np.isnan(observed.x_train[:, 0]).astype(float),
        }
    )
    data["bin"] = pd.qcut(data["anchor"], q=8, duplicates="drop")
    aggregate = data.groupby("bin", observed=False).agg(
        mean_anchor=("anchor", "mean"),
        missing_rate=("missing_X1", "mean"),
    )
    ax.plot(
        aggregate["mean_anchor"], aggregate["missing_rate"],
        marker="o", label=name.upper()
    )
ax.set(
    title="Conditional missingness for predictor X1",
    xlabel="Observed X3 (bin average)",
    ylabel="Fraction of X1 unobserved",
)
ax.legend()
fig.tight_layout()
plt.show()

## 2. Estimators and evaluation discipline

Every model uses a median imputer with missingness indicators fitted *only* to its training portion, then standardisation.

- OLS, ridge and elastic net represent familiar linear baselines.
- Huber regression reduces sensitivity to extreme training outcomes.
- PCR retains 95% of available imputed-X variance.
- PLS uses two supervised components.

An internal train/validation split then selects a candidate once by **observed validation RMSE**, and independently by **observed validation net benefit**. The winner is refitted on the full training set.

This is a single-split selection experiment, not a nested-repeated validation claim. In response-outlier scenarios, the validation target itself can be contaminated and distort either selection criterion.

In [ ]:
experiment = run_stress_experiment(
    repeats=REPEATS,
    n_train_values=TRAIN_SIZES,
    n_test=500,
    seed=2026,
    cost=COST,
)
summary = (
    experiment.groupby(["signal", "n_train", "stress", "model"], as_index=False)
    .agg(
        mean_rmse=("rmse", "mean"),
        mean_regret=("regret", "mean"),
        mean_profit=("expected_profit", "mean"),
    )
)
middle_size = min(TRAIN_SIZES, key=lambda n: abs(n - 180))
display(
    summary.loc[
        (summary["signal"] == "low_variance_signal")
        & (summary["n_train"] == middle_size)
        & summary["stress"].isin(["clean", "mar", "outliers"])
    ]
    .sort_values(["stress", "mean_regret"])
    .round(3)
)

## 3. Paired changes from clean-data regret

Rather than comparing unrelated simulated samples, we compute the difference between each stressed replicate and its clean counterpart. The matching unit is a simulated dataset, indexed by response mechanism, training size, replicate and estimator.

We bootstrap the **paired replicate differences** to describe Monte Carlo uncertainty. These intervals are not confidence statements about real-world populations, and very few replicates cannot support useful uncertainty estimates.

In [ ]:
if REPEATS >= 3:
    paired = paired_regret_shift(experiment, bootstrap_draws=500, seed=37)
    display(
        paired.loc[
            (paired["signal"] == "low_variance_signal")
            & (paired["n_train"] == middle_size)
            & paired["stress"].isin(["mcar", "mar", "outliers", "mar_outliers"])
        ]
        .sort_values(["stress", "paired_mean_delta_regret"])
        .round(3)
    )
else:
    print("Smoke run only: too few replicates for useful bootstrap intervals.")
    paired = None

In [ ]:
subset = summary.loc[
    (summary["signal"] == "low_variance_signal")
    & (summary["n_train"] == middle_size)
]
pivot = subset.pivot(index="model", columns="stress", values="mean_regret")
pivot = pivot.reindex(columns=list(STRESSES))
fig, ax = plt.subplots(figsize=(10, 4.5))
image = ax.imshow(pivot.to_numpy(), aspect="auto")
ax.set_xticks(np.arange(len(pivot.columns)), labels=pivot.columns, rotation=25)
ax.set_yticks(np.arange(len(pivot.index)), labels=pivot.index)
ax.set(title=f"Mean regret by stress, low-variance signal, n={middle_size}")
fig.colorbar(image, ax=ax, label="Expected regret per test observation")
fig.tight_layout()
plt.show()

## 4. Does optimising a validation metric select the same estimator?

The two validation selection policies score the same candidate methods on the same held-out training observations, but target different objectives. Their selections need not agree.

An observed utility estimate is noisy and may be particularly unreliable if validation labels contain gross errors. Avoid concluding that utility-guided selection is universally better, even when its test-regret ranking is favourable in one case.

In [ ]:
selection = experiment.loc[
    experiment["model"].isin(["Validation utility", "Validation RMSE"]),
    ["signal", "n_train", "stress", "replicate", "model", "selected_candidate"],
]
display(
    selection.groupby(["stress", "model", "selected_candidate"])
    .size()
    .rename("times_selected")
    .reset_index()
    .sort_values(["stress", "model", "times_selected"], ascending=[True, True, False])
)
display(
    summary.loc[
        (summary["model"].isin(["Validation utility", "Validation RMSE"]))
        & (summary["n_train"] == middle_size)
    ]
    .sort_values(["signal", "stress", "model"])
    .round(3)
)

## 5. Interpretation

A statistically significant coefficient, a high-variance component, a stable prediction and an economically valuable decision remain different objects.

The experimental comparisons are meaningful only under explicit assumptions:

1. The MAR process uses an observed predictor \(X_3\); it is **not MNAR**, and no universal missingness correction is claimed.
2. Median imputation with indicators is a defensible transparent benchmark, not an unbiased estimator under arbitrary MAR.
3. The oracle uses **unobserved full-data predictors** and is therefore unattainable in missingness cases. Its regret measures both irreducible missing-information loss and model error.
4. Outcome contamination is restricted to training labels; test labels and underlying conditional means remain unchanged. It models measurement errors, not concept drift.
5. Our synthetic conditional benefit is **not identified causal treatment benefit** in observational data.
6. Model selection uses a single internal split. Repeating it or nesting cross-validation is a necessary sensitivity analysis before stronger conclusions.

### Future experiments

Investigate explicit missingness-model estimators, multiple imputation, MNAR sensitivity, strong predictor contamination, changes in \(p/n\), nonlinear interactions, uncertainty in the cost threshold, drift across training and deployment and model selection using causal welfare estimates.

For the original statistical motivation, read the first notebook, *evidence-review.ipynb*, in the same folder.